<div align="center">

# <span style="color:#2E86C1;"> 1 - Data Preparation for Machine Learning</span>

**Author:** [<span style="color:#8E44AD;">Dr. D Bhanu Prakash</span>](https://dbhanuprakash233.github.io)

<img src="https://github.com/dbhanuprakash233/SSSIHL_DBP/blob/main/assets/SssihlLogo.jpeg?raw=true" alt="University Logo" width="80"/>

**<span style="color:#16A085;">Sri Sathya Sai Institute of Higher Learning</span>**  
<span style="color:#5D6D7E;">Prasanthi Nilayam - 515 134, Andhra Pradesh, India.</span>

**Course:** <span style="color:#D35400;">Data Analysis and Visualization</span>  
**Course Code:** <span style="color:#1ABC9C;">UDSC-402</span>

</div>

#### About this notebook

This notebook contains the **complete lecture notes** (all text, definitions, taxonomies and
tables preserved verbatim) with **runnable Python code** added at every point where a method
is described. Three primary datasets are used throughout so that the code tells one continuous
story instead of jumping between toy examples:

| Dataset | Rows x Cols | Task | Used for |
|---|---|---|---|
| **Horse Colic** | 300 x 28 | Binary classification | Missing data, imputation (Ch. 7-10), leakage (Ch. 4) |
| **Boston Housing** | 506 x 14 | Regression | Outliers (Ch. 6), transforms (Ch. 3) |
| *Oil Spill* (supporting) | 937 x 50 | Binary classification | Basic cleaning: constant / near-constant columns (Ch. 5) |

The final section exports this notebook to a **styled PDF with configurable page borders
(margins) and font styles**, driven from a single configuration dictionary.

## Section 0 — Environment Setup

In [ ]:
# If anything is missing, uncomment and run:
# !pip install numpy pandas scikit-learn matplotlib seaborn

import os, sys, warnings, io, urllib.request
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

print("python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Dataset loader: local cache -> GitHub download -> synthetic fallback
# ---------------------------------------------------------------------------
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"

def load_dataset(name, na_values=None):
    # name: one of 'horse-colic', 'housing', 'oil-spill'
    fname = os.path.join(DATA_DIR, name + ".csv")
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(BASE_URL + name + ".csv", fname)
            print(f"[downloaded] {name}.csv")
        except Exception as e:
            print(f"[offline] could not download {name}.csv ({e}) -> synthetic fallback")
            return _synthetic(name)
    return pd.read_csv(fname, header=None, na_values=na_values)

def _synthetic(name):
    # Small stand-ins with the SAME structural quirks, so all code below still runs.
    rng = np.random.default_rng(RANDOM_STATE)
    if name == "horse-colic":
        df = pd.DataFrame(rng.normal(size=(300, 28)).round(2))
        for c in [3, 5, 7, 13, 15, 20]:                      # inject missing values
            df.loc[rng.choice(300, size=int(300 * 0.2), replace=False), c] = np.nan
        df[23] = rng.integers(1, 3, size=300)                 # target column 23 (1/2)
        return df
    if name == "housing":
        df = pd.DataFrame(rng.normal(size=(506, 14)).round(3))
        df[13] = 20 + 5 * rng.normal(size=506)                # target = MEDV
        return df
    if name == "oil-spill":
        df = pd.DataFrame(rng.normal(size=(937, 50)).round(3))
        df[22] = 0                                            # constant column
        df[[3, 8, 30]] = rng.integers(0, 3, size=(937, 3))    # near-constant columns
        df[49] = (rng.random(937) < 0.05).astype(int)         # imbalanced target
        return df
    raise ValueError(name)

# --- Load the three datasets --------------------------------------------------
# Horse Colic: '?' is the sentinel for missing values. We deliberately load it RAW
# (sentinels intact) here, because Chapter 7 is about marking them.
horse_raw   = load_dataset("horse-colic")                  # '?' still present
horse       = load_dataset("horse-colic", na_values="?")   # '?' -> NaN
housing     = load_dataset("housing")
oil         = load_dataset("oil-spill")

HORSE_TARGET, OIL_TARGET, HOUSING_TARGET = 23, 49, 13

print("horse colic :", horse.shape)
print("housing     :", housing.shape)
print("oil spill   :", oil.shape)

In [ ]:
# Convenience splitters used repeatedly below.
def xy(df, target_col):
    # Returns X (all columns except target) and y (target) as numpy arrays.
    cols = [c for c in df.columns if c != target_col]
    return df[cols].values, df[target_col].values

X_horse, y_horse = xy(horse, HORSE_TARGET)          # contains NaN on purpose
X_house, y_house = xy(housing, HOUSING_TARGET)
X_oil,   y_oil   = xy(oil, OIL_TARGET)

HOUSING_NAMES = ["CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS",
                 "RAD","TAX","PTRATIO","B","LSTAT","MEDV"]
housing.columns = HOUSING_NAMES   # readable names for the housing frame

print("X_horse", X_horse.shape, "| NaNs:", int(np.isnan(X_horse.astype(float)).sum()))
print("X_house", X_house.shape, "| y range:", round(y_house.min(),1), "-", round(y_house.max(),1))
print("X_oil  ", X_oil.shape,   "| class balance:", np.bincount(y_oil.astype(int)))

---
# Chapter 1 — Data Preparation in a Machine Learning Project

## The Applied Machine Learning Process (4 Steps)

Every predictive modeling project — regardless of the specific dataset — follows the same
general four-step process:

- **Step 1: Define Problem** — frame the task (classification, regression, or another problem
  type), collect the relevant data, and understand it through summary statistics and visualization.
- **Step 2: Prepare Data** — transform the raw, collected data into a form usable for modeling.
- **Step 3: Evaluate Models** — design a robust test harness: choose a performance metric,
  establish a baseline, choose a resampling technique (train-test split or k-fold
  cross-validation), and perform hyperparameter tuning / ensembling.
- **Step 4: Finalize Model** — select the final model (model selection), evaluate on a hold-out
  validation set, summarize performance for stakeholders, and productionize the model.

## Key Definition

> ***Data Preparation:*** The transformation of raw data into a form that is more suitable for
> modeling. Also referred to as data wrangling, data munging, data cleaning, data pre-processing,
> or (loosely) feature engineering.

## Why Raw Data Cannot Be Used Directly

- Machine learning algorithms require input data to be numbers.
- Some algorithms impose specific requirements on the data (e.g., a particular probability
  distribution, no correlated inputs).
- Statistical noise and errors in the data may need to be corrected.
- Complex nonlinear relationships in the data may need to be exposed/teased out.

## The Five Core Data Preparation Tasks (detailed fully in Chapter 3)

- **Data Cleaning** — identifying and correcting mistakes or errors in the data.
- **Feature Selection** — identifying the input variables most relevant to the prediction task.
- **Data Transforms** — changing the scale or distribution of variables.
- **Feature Engineering** — deriving new input variables from the available data.
- **Dimensionality Reduction** — creating compact, lower-dimensional projections of the data.

*Guiding philosophy: data preparation should best expose the unknown underlying structure of the
prediction problem to the learning algorithm — this is why it is treated as a step of discovery,
not a fixed recipe.*

### Code — Step 1 in practice: *define the problem* and *understand the data*

Summary statistics + visualization on the two primary datasets.

In [ ]:
# STEP 1 (a): frame the task and inspect the raw data ------------------------
print("=== HORSE COLIC — binary classification (target col 23: 1 = surgical lesion, 2 = not) ===")
print(horse.shape)
display(horse.head())
print("\nClass distribution:\n", horse[HORSE_TARGET].value_counts().to_string())

print("\n=== BOSTON HOUSING — regression (target MEDV: median home value in $1000s) ===")
display(housing.head())

In [ ]:
# STEP 1 (b): summary statistics --------------------------------------------
print("Housing — descriptive statistics (note the wildly different scales per column):")
display(housing.describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2))

In [ ]:
# STEP 1 (c): visualization --------------------------------------------------
housing[["CRIM","RM","AGE","DIS","LSTAT","MEDV"]].hist(bins=25, figsize=(10, 5))
plt.suptitle("Housing — distributions (Step 1: understand the data)")
plt.tight_layout(); plt.show()

# Missing-data map for horse colic (white = missing)
plt.figure(figsize=(9, 3))
plt.imshow(horse.isnull().T, aspect="auto", cmap="gray_r", interpolation="nearest")
plt.xlabel("row index"); plt.ylabel("column index")
plt.title("Horse Colic — missing value map (dark = missing)")
plt.tight_layout(); plt.show()

In [ ]:
# STEPS 2-4 in miniature: prepare -> evaluate -> finalize --------------------
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split

# Step 2: Prepare  (impute + scale, wrapped in a Pipeline so it is leakage-safe)
pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("scale",  MinMaxScaler()),
    ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
])

# Step 3: Evaluate (metric + resampling strategy + baseline)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
scores = cross_val_score(pipe, X_horse, y_horse, scoring="accuracy", cv=cv, n_jobs=-1)
print(f"Pipeline accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

from sklearn.dummy import DummyClassifier
base = cross_val_score(DummyClassifier(strategy="most_frequent"), X_horse, y_horse, cv=cv, n_jobs=-1)
print(f"Baseline accuracy: {base.mean():.3f}  <- must beat this to be useful")

# Step 4: Finalize (fit on all training data, evaluate on a held-out set, then save)
X_tr, X_te, y_tr, y_te = train_test_split(X_horse, y_horse, test_size=0.30,
                                          stratify=y_horse, random_state=RANDOM_STATE)
pipe.fit(X_tr, y_tr)
print(f"Hold-out accuracy: {pipe.score(X_te, y_te):.3f}")
# import joblib; joblib.dump(pipe, 'final_pipeline.joblib')   # productionize: save transforms + model TOGETHER

---
# Chapter 2 — Why Data Preparation Is So Important

> ***Data (in an ML context):*** Rows of observations and columns of variables
> (structured/tabular data), where a machine learning algorithm learns a mapping from the input
> variables to a target (output) variable.

## Three Reasons Data Preparation Matters

- **Machine Learning Algorithms Expect Numbers** — text, categories, and dates must be encoded
  numerically before modeling.
- **Machine Learning Algorithms Have Requirements** — e.g., a Gaussian-distributed input, the
  absence of missing values, or uncorrelated input features.
- **Model Performance Depends on Data** — the chosen representation of data (scaling, engineered
  features) can influence performance as much as, or more than, the choice of algorithm itself.

## Key Takeaway

Predictive modeling is, in practice, mostly data preparation. Practitioners typically spend the
majority of a project's time and effort on this step because it directly determines how well the
underlying structure of a problem is exposed to the learning algorithm.

### Code — Reason 1: algorithms expect numbers (watch it fail, then fix it)

In [ ]:
from sklearn.linear_model import LogisticRegression

toy = pd.DataFrame({
    "colour": ["red", "green", "blue", "green", "red", "blue"],
    "size":   ["S", "M", "L", "M", "L", "S"],
    "label":  [0, 1, 0, 1, 1, 0],
})

# (a) FAILS: raw text handed straight to an estimator
try:
    LogisticRegression().fit(toy[["colour", "size"]], toy["label"])
except Exception as e:
    print("ERROR ->", type(e).__name__, ":", str(e).split("\n")[0])

# (b) WORKS: encode categories as numbers first (Chapter 3 transforms)
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder()
X_enc = enc.fit_transform(toy[["colour", "size"]])
LogisticRegression().fit(X_enc, toy["label"])
print("\nEncoded input:\n", X_enc)
print("Fit succeeded once the data was numeric.")

In [ ]:
# Reason 2: algorithms have requirements (e.g. NO missing values) -----------
from sklearn.linear_model import LinearRegression
try:
    LogisticRegression().fit(X_horse, y_horse)          # X_horse still contains NaN
except Exception as e:
    print("ERROR ->", type(e).__name__, ":", str(e).split("\n")[0][:110])
print("\n=> This is exactly why Chapters 7-10 exist.")

In [ ]:
# Reason 3: model performance depends on the DATA REPRESENTATION ------------
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

raw_pipe    = Pipeline([("impute", SimpleImputer()), ("model", KNeighborsClassifier())])
scaled_pipe = Pipeline([("impute", SimpleImputer()), ("scale", StandardScaler()),
                        ("model", KNeighborsClassifier())])

for name, p in [("KNN on raw values      ", raw_pipe), ("KNN on standardized data", scaled_pipe)]:
    s = cross_val_score(p, X_horse, y_horse, scoring="accuracy", cv=cv, n_jobs=-1)
    print(f"{name}: {s.mean():.3f}")

print("\nSame algorithm, same data, different representation -> different performance.")

---
# Chapter 3 — Tour of Data Preparation Techniques (Core Taxonomy)

**This chapter is the backbone of the subject — memorize this taxonomy carefully.**

## 3.1 Data Cleaning

> ***Data Cleaning:*** Fixing systematic problems or errors in messy data, typically guided by
> domain expertise.

- Using statistics to define "normal" data and identify outliers (Ch. 6).
- Identifying & removing columns with the same value / zero variance, and duplicate rows (Ch. 5).
- Marking empty values as missing (Ch. 7).
- Imputing missing values using statistics or a learned model (Ch. 8, 9, 10).

Data cleaning is typically performed first, before other data preparation steps.

In [ ]:
# 3.1 — a one-glance data-cleaning audit you can reuse on any new dataset
def cleaning_audit(df, name="dataset"):
    n_rows = len(df)
    rep = pd.DataFrame({
        "dtype":        df.dtypes.astype(str),
        "n_unique":     df.nunique(),
        "pct_unique":   (df.nunique() / n_rows * 100).round(2),
        "n_missing":    df.isnull().sum(),
        "pct_missing":  (df.isnull().mean() * 100).round(2),
        "variance":     df.var(numeric_only=True).round(4),
    })
    print(f"--- {name}: {df.shape[0]} rows x {df.shape[1]} cols | duplicate rows: {df.duplicated().sum()} ---")
    print(f"    constant (zero-variance) columns : {list(rep.index[rep.n_unique == 1])}")
    print(f"    near-constant (<1% unique)       : {list(rep.index[rep.pct_unique < 1])}")
    print(f"    columns with missing values      : {list(rep.index[rep.n_missing > 0])}")
    return rep

audit = cleaning_audit(oil, "OIL SPILL")
display(audit.head(12))
_ = cleaning_audit(horse, "HORSE COLIC")

## 3.2 Feature Selection

> ***Feature Selection:*** Techniques for selecting a subset of input features that are most
> relevant to the target variable, to avoid misleading the model with irrelevant/redundant inputs
> and to favor simpler models.

**Taxonomy of feature selection methods:**

- **Unsupervised** — does not use the target variable (e.g., removing highly correlated inputs).
- **Supervised** — uses the target variable, and is further divided into:
  - **Intrinsic** — feature selection happens automatically as part of model fitting
    (e.g., LASSO regression, decision trees).
  - **Wrapper** — explicitly searches for the subset of features that produces the
    best-performing model (e.g., Recursive Feature Elimination, RFE).
  - **Filter** — scores each input feature independently (typically with a statistical measure
    such as correlation) and selects the highest-scoring subset.

In [ ]:
# --- UNSUPERVISED: drop highly correlated inputs (target never used) --------
corr = housing.drop(columns=["MEDV"]).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.75).any()]
print("Highly correlated (|r| > 0.75) columns that could be dropped:", to_drop)

plt.figure(figsize=(6.5, 5))
plt.imshow(corr, cmap="viridis"); plt.colorbar(label="|correlation|")
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr)), corr.columns, fontsize=7)
plt.title("Unsupervised feature selection: correlation matrix")
plt.tight_layout(); plt.show()

In [ ]:
# --- FILTER: score each feature independently, keep the best k --------------
from sklearn.feature_selection import SelectKBest, f_regression, f_classif, mutual_info_classif

Xh = housing.drop(columns=["MEDV"]).values
yh = housing["MEDV"].values
sel = SelectKBest(score_func=f_regression, k=5).fit(Xh, yh)
scores = pd.Series(sel.scores_, index=housing.columns[:-1]).sort_values(ascending=False)
print("ANOVA F-scores (filter method) — housing:\n", scores.round(1).to_string())
print("\nTop-5 selected:", list(scores.index[:5]))

plt.bar(scores.index, scores.values); plt.xticks(rotation=90)
plt.ylabel("F-score"); plt.title("Filter feature selection"); plt.tight_layout(); plt.show()

In [ ]:
# --- WRAPPER: Recursive Feature Elimination (RFE) ---------------------------
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

rfe_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("rfe",    RFE(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), n_features_to_select=8)),
    ("model",  DecisionTreeClassifier(random_state=RANDOM_STATE)),
])
s = cross_val_score(rfe_pipe, X_horse, y_horse, scoring="accuracy", cv=cv, n_jobs=-1)
print(f"RFE (wrapper) + tree accuracy on horse colic: {s.mean():.3f}")

rfe_pipe.fit(X_horse, y_horse)
mask = rfe_pipe.named_steps["rfe"].support_
kept = [c for c, m in zip([c for c in horse.columns if c != HORSE_TARGET], mask) if m]
print("Columns kept by RFE:", kept)

In [ ]:
# --- INTRINSIC: selection happens as part of model fitting ------------------
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

lasso = Pipeline([("scale", StandardScaler()), ("model", Lasso(alpha=1.0))]).fit(Xh, yh)
coef = pd.Series(lasso.named_steps["model"].coef_, index=housing.columns[:-1])
print("LASSO coefficients (exact zeros = feature removed by the model itself):")
print(coef.round(3).to_string())
print("\nDropped by LASSO:", list(coef.index[coef == 0]))

rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE).fit(Xh, yh)
imp = pd.Series(rf.feature_importances_, index=housing.columns[:-1]).sort_values(ascending=False)
print("\nDecision-tree/forest importances (intrinsic):\n", imp.round(3).to_string())

## 3.3 Data Transforms

> ***Data Transform:*** A technique that changes the type or the probability distribution of a
> data variable.

**Data type taxonomy:**

- **Numeric Data Type:** Integer (no fractional part), Float (floating point values).
- **Categorical Data Type:** Ordinal (ranked labels), Nominal (unranked labels),
  Boolean (True/False).

| Transform | Definition |
|---|---|
| **Discretization Transform** | Encode a numeric variable as an ordinal variable. |
| **Ordinal Transform** | Encode a categorical variable into an integer variable. |
| **One Hot Transform** | Encode a categorical variable into binary (dummy) variables. |
| **Normalization Transform** | Scale a variable to the range 0 to 1. |
| **Standardization Transform** | Scale a variable to a standard Gaussian (mean = 0, std = 1). |
| **Power Transform** | Change the distribution of a variable to be more Gaussian. |
| **Quantile Transform** | Impose a probability distribution (uniform or Gaussian) on a variable. |

**Important:** transforms are generally fit and applied **per variable**, and the fitted transform
object must be **saved (along with the final model)** to correctly process new/future data.

In [ ]:
# ---- All seven transforms from the table, on one column of the housing data
from sklearn.preprocessing import (KBinsDiscretizer, OrdinalEncoder, OneHotEncoder,
                                   MinMaxScaler, StandardScaler, PowerTransformer,
                                   QuantileTransformer)

col = housing[["CRIM"]].values          # heavily skewed -> good demo column

# 1. DISCRETIZATION: numeric -> ordinal
disc = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="uniform")
crim_bins = disc.fit_transform(col)

# 2. ORDINAL TRANSFORM: categorical -> integer
cats = pd.DataFrame({"size": ["S", "M", "L", "M", "S", "L"]})
ordinal = OrdinalEncoder(categories=[["S", "M", "L"]])       # ranked order matters
size_int = ordinal.fit_transform(cats)

# 3. ONE HOT TRANSFORM: categorical -> binary dummies
onehot = OneHotEncoder(sparse_output=False)
size_dummies = onehot.fit_transform(cats)

# 4. NORMALIZATION: scale to [0, 1]
crim_norm = MinMaxScaler().fit_transform(col)

# 5. STANDARDIZATION: mean 0, std 1
crim_std = StandardScaler().fit_transform(col)

# 6. POWER TRANSFORM: make more Gaussian (Yeo-Johnson handles zeros/negatives)
crim_pow = PowerTransformer(method="yeo-johnson").fit_transform(col)

# 7. QUANTILE TRANSFORM: impose a distribution
crim_qtl = QuantileTransformer(n_quantiles=100, output_distribution="normal",
                               random_state=RANDOM_STATE).fit_transform(col)

print("1. Discretized  :", crim_bins[:5].ravel())
print("2. Ordinal      :", size_int.ravel())
print("3. One-hot      :\n", size_dummies)
print("4. Normalized   :", crim_norm[:5].ravel().round(4))
print("5. Standardized :", crim_std[:5].ravel().round(4))
print("6. Power        :", crim_pow[:5].ravel().round(4))
print("7. Quantile     :", crim_qtl[:5].ravel().round(4))

In [ ]:
# Visual comparison of the distribution-changing transforms
fig, ax = plt.subplots(1, 4, figsize=(13, 3))
for a, (data, title) in zip(ax, [(col, "Original (skewed)"),
                                 (crim_std, "Standardized"),
                                 (crim_pow, "Power (Yeo-Johnson)"),
                                 (crim_qtl, "Quantile -> normal")]):
    a.hist(data, bins=30); a.set_title(title, fontsize=9)
plt.suptitle("Chapter 3.3 — the same variable under different transforms")
plt.tight_layout(); plt.show()

In [ ]:
# CRITICAL: a fitted transform is a stateful object — SAVE IT WITH THE MODEL.
scaler = MinMaxScaler().fit(housing[["CRIM"]])
print("Learned state -> data_min_:", scaler.data_min_, " data_max_:", scaler.data_max_)
print("New unseen row [0.5] scales to:", scaler.transform([[0.5]]).ravel())

# import joblib
# joblib.dump({'scaler': scaler, 'model': rf}, 'model_bundle.joblib')   # both, together

## 3.4 Feature Engineering

> ***Feature Engineering:*** The process of creating new input variables from the available data,
> typically requiring subject-matter expertise.

- Adding a boolean flag variable for some state.
- Adding a group or global summary statistic (e.g., a mean).
- Adding new variables for each component of a compound variable (e.g., splitting a date-time
  into year/month/day).
- **Polynomial Transform** — creating copies of numerical input variables raised to a power
  (or multiplied together).

In [ ]:
fe = housing.copy()

# (a) Boolean flag variable for some state
fe["IS_RIVERSIDE"]   = (fe["CHAS"] == 1).astype(int)
fe["HIGH_CRIME"]     = (fe["CRIM"] > fe["CRIM"].quantile(0.90)).astype(int)

# (b) Group / global summary statistic
fe["TAX_BRACKET"]    = pd.qcut(fe["TAX"], 4, labels=False)
fe["RM_VS_GRP_MEAN"] = fe["RM"] - fe.groupby("TAX_BRACKET")["RM"].transform("mean")
fe["LSTAT_VS_GLOBAL_MEAN"] = fe["LSTAT"] - fe["LSTAT"].mean()

# (c) Split a compound variable into components (date-time example)
dates = pd.DataFrame({"ts": pd.to_datetime(["2024-01-15 08:30", "2024-07-04 19:45",
                                            "2025-11-30 23:05"])})
dates["year"], dates["month"], dates["day"] = dates.ts.dt.year, dates.ts.dt.month, dates.ts.dt.day
dates["hour"], dates["dayofweek"], dates["is_weekend"] = (dates.ts.dt.hour, dates.ts.dt.dayofweek,
                                                          dates.ts.dt.dayofweek.isin([5, 6]).astype(int))
display(dates)
display(fe[["CRIM","HIGH_CRIME","RM","TAX_BRACKET","RM_VS_GRP_MEAN","LSTAT_VS_GLOBAL_MEAN"]].head())

In [ ]:
# (d) POLYNOMIAL TRANSFORM: powers and pairwise products of the inputs
from sklearn.preprocessing import PolynomialFeatures

sub = housing[["RM", "LSTAT", "NOX"]]
poly = PolynomialFeatures(degree=2, include_bias=False)
Xp = poly.fit_transform(sub)
print("before:", sub.shape, "-> after:", Xp.shape)
print("new feature names:", list(poly.get_feature_names_out(sub.columns)))

# Does the engineered representation actually help?
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
plain = cross_val_score(LinearRegression(), sub.values, yh, scoring="neg_mean_absolute_error", cv=kf)
polyp = cross_val_score(Pipeline([("poly", PolynomialFeatures(2, include_bias=False)),
                                  ("m", LinearRegression())]),
                        sub.values, yh, scoring="neg_mean_absolute_error", cv=kf)
print(f"\nMAE plain      : {-plain.mean():.3f}")
print(f"MAE polynomial : {-polyp.mean():.3f}")

## 3.5 Dimensionality Reduction

> ***Dimensionality:*** The number of input features for a dataset; more input variables define a
> higher-dimensional feature space.

> ***Curse of Dimensionality:*** As the number of dimensions (features) increases, the data
> becomes an increasingly sparse and unrepresentative sample of that space.

> ***Dimensionality Reduction:*** Creating a projection of the data into a lower-dimensional space
> that preserves the most important properties of the original data; unlike feature selection,
> projected variables are **not directly interpretable** in terms of the original inputs.

**Matrix-factorization based methods:**

- Principal Component Analysis (PCA)
- Singular Value Decomposition (SVD)

**Model-based methods:**

- Linear Discriminant Analysis (LDA)
- Autoencoders

**Manifold learning methods:**

- Self-Organizing Maps (SOM)
- t-Distributed Stochastic Neighbor Embedding (t-SNE)

The main effect of matrix-factorization methods is **removing linear dependencies (correlations)**
between input variables.

In [ ]:
# --- Matrix factorization: PCA and SVD --------------------------------------
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.manifold import TSNE

X_oil_std = StandardScaler().fit_transform(X_oil)

pca = PCA(n_components=10).fit(X_oil_std)
print("PCA explained variance ratio (first 10):", pca.explained_variance_ratio_.round(3))
print("Cumulative:", pca.explained_variance_ratio_.cumsum().round(3))

svd = TruncatedSVD(n_components=10, random_state=RANDOM_STATE).fit(X_oil_std)
print("SVD explained variance ratio (first 10):", svd.explained_variance_ratio_.round(3))

plt.plot(np.arange(1, 11), pca.explained_variance_ratio_.cumsum(), marker="o")
plt.xlabel("number of components"); plt.ylabel("cumulative explained variance")
plt.title("PCA scree — oil spill (49 inputs)"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# PCA removes linear dependencies: correlations between components are ~0
Z = PCA(n_components=6).fit_transform(X_oil_std)
print("Max |correlation| between original inputs :",
      round(np.abs(np.corrcoef(X_oil_std.T) - np.eye(X_oil_std.shape[1])).max(), 3))
print("Max |correlation| between PCA components  :",
      round(np.abs(np.corrcoef(Z.T) - np.eye(Z.shape[1])).max(), 6))

In [ ]:
# --- Model-based: LDA  |  Manifold: t-SNE  (SOM/autoencoders need extra libs)
lda_proj = LDA(n_components=1).fit_transform(X_oil_std, y_oil)     # n_components <= n_classes-1
tsne_proj = TSNE(n_components=2, perplexity=30, init="pca",
                 random_state=RANDOM_STATE).fit_transform(X_oil_std)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].scatter(Z[:, 0], Z[:, 1], c=y_oil, s=8, cmap="coolwarm"); ax[0].set_title("PCA (2 comps)")
ax[1].scatter(lda_proj.ravel(), np.random.normal(0,.1,len(lda_proj)), c=y_oil, s=8, cmap="coolwarm")
ax[1].set_title("LDA (1 comp, jittered)")
ax[2].scatter(tsne_proj[:,0], tsne_proj[:,1], c=y_oil, s=8, cmap="coolwarm"); ax[2].set_title("t-SNE")
plt.suptitle("Chapter 3.5 — projections of the 49-dimensional oil-spill data")
plt.tight_layout(); plt.show()

# Autoencoder (model-based, sketch): a neural net trained to reconstruct its own input;
# the bottleneck layer is the reduced representation. Needs TensorFlow/PyTorch:
#   encoder = Dense(bottleneck)(input); decoder = Dense(n_inputs)(encoder)
# Self-Organizing Map (SOM): available via the 'minisom' package.

In [ ]:
# Does dimensionality reduction help the downstream model? Test it in a Pipeline.
from sklearn.linear_model import LogisticRegression
for k in [None, 5, 10, 20]:
    steps = [("scale", StandardScaler())]
    if k: steps.append(("pca", PCA(n_components=k)))
    steps.append(("model", LogisticRegression(max_iter=1000)))
    s = cross_val_score(Pipeline(steps), X_oil, y_oil, scoring="roc_auc", cv=cv, n_jobs=-1)
    print(f"PCA components = {str(k):>4} -> ROC AUC {s.mean():.3f}")

---
# Chapter 4 — Data Preparation Without Data Leakage

> ***Data Leakage:*** A problem where information about the holdout (test/validation) dataset is
> made available to the model during training, giving it an unrealistic advantage and producing an
> overly optimistic or otherwise incorrect estimate of model performance.

## The Naive (INCORRECT) Approach

1. Prepare / transform the **entire** dataset (e.g., normalize using the global min and max of all rows).
2. Split the data into train and test sets.
3. Evaluate the model.

This causes **indirect data leakage**: summary statistics (min, max, mean, standard deviation)
computed over the whole dataset leak information about the test set into the training process —
even though the test rows are never directly used to fit the model.

## The Correct Approach — Train/Test Split

1. Split the data into train and test sets **FIRST**.
2. Fit the data-preparation transform (e.g., a scaler) on the **TRAINING set only**.
3. Apply the fitted transform to both the training and test sets.
4. Evaluate the model.

## The Correct Approach — k-Fold Cross-Validation

Data preparation must be fit **within each fold**, using only that fold's training portion, then
applied to that fold's held-out portion. In practice this is achieved with a **Pipeline** (an
ordered sequence of transform steps ending in a model), which is passed as a single unit to the
cross-validation function (e.g., `cross_val_score()`).

*Key principle: "The entire modeling pipeline must be prepared only on the training dataset" — so
this kind of evaluation is really **pipeline evaluation**, not merely model evaluation.*

## k-Fold Cross-Validation — Recap

Split the dataset into k non-overlapping folds. Train the model on k−1 folds and test it on the
remaining (held-out) fold. Repeat so every fold is used as the test set exactly once, and report
the average performance across all folds.

> ***Repeated Stratified k-Fold:*** Cross-validation repeated multiple times (for robustness),
> where each fold preserves the overall class ratio of the dataset (stratified) — considered best
> practice for classification problems.

k-fold cross-validation generally gives a more reliable performance estimate than a single
train-test split, at the cost of higher computation (repeated model fitting).

In [ ]:
# ---- NAIVE (WRONG): transform the whole dataset, then split ---------------
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

X_syn, y_syn = make_classification(n_samples=1000, n_features=20, n_informative=15,
                                   n_redundant=5, random_state=RANDOM_STATE)

X_all_scaled = MinMaxScaler().fit_transform(X_syn)      # <-- min/max seen the TEST rows: LEAKAGE
Xtr, Xte, ytr, yte = train_test_split(X_all_scaled, y_syn, test_size=.33, random_state=1)
m = LogisticRegression().fit(Xtr, ytr)
print(f"NAIVE  (leaky)   accuracy: {accuracy_score(yte, m.predict(Xte))*100:.3f}%")

In [ ]:
# ---- CORRECT: split first, fit the transform on TRAIN only ----------------
Xtr, Xte, ytr, yte = train_test_split(X_syn, y_syn, test_size=.33, random_state=1)
sc = MinMaxScaler().fit(Xtr)             # statistics learned from training rows only
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
m = LogisticRegression().fit(Xtr_s, ytr)
print(f"CORRECT (train/test split) accuracy: {accuracy_score(yte, m.predict(Xte_s))*100:.3f}%")
print("\nThe naive score is optimistic — it is not an honest estimate of future performance.")

In [ ]:
# ---- CORRECT: k-fold CV with a Pipeline (transform re-fit inside every fold)
from sklearn.model_selection import RepeatedStratifiedKFold

cv10 = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

# WRONG: scale everything up-front, then cross-validate
leaky = cross_val_score(LogisticRegression(), MinMaxScaler().fit_transform(X_syn), y_syn,
                        scoring="accuracy", cv=cv10, n_jobs=-1)

# RIGHT: the scaler lives INSIDE the pipeline, so it is re-fit on each fold's training part
pipe_ok = Pipeline([("scale", MinMaxScaler()), ("model", LogisticRegression())])
clean = cross_val_score(pipe_ok, X_syn, y_syn, scoring="accuracy", cv=cv10, n_jobs=-1)

print(f"Leaky CV accuracy   : {leaky.mean()*100:.3f}% (+/- {leaky.std()*100:.3f})")
print(f"Pipeline CV accuracy: {clean.mean()*100:.3f}% (+/- {clean.std()*100:.3f})   <- trustworthy")

In [ ]:
# k-fold mechanics, made explicit (what cross_val_score does internally)
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for i, (tr_idx, te_idx) in enumerate(kf.split(X_syn), 1):
    p = Pipeline([("scale", MinMaxScaler()), ("model", LogisticRegression())])
    p.fit(X_syn[tr_idx], y_syn[tr_idx])            # transform fit on THIS fold's train part only
    print(f"fold {i}: train={len(tr_idx):4d} test={len(te_idx):4d} acc={p.score(X_syn[te_idx], y_syn[te_idx]):.3f}")

# Stratification preserves the class ratio in every fold
strat = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
ratios = [np.bincount(y_oil[te].astype(int))[1] / len(te) for _, te in strat.split(X_oil, y_oil)]
print("\nMinority-class ratio per stratified fold (oil spill, overall %.3f): %s"
      % (y_oil.mean(), np.round(ratios, 3)))

---
# Chapter 5 — Basic Data Cleaning

Always run these basic checks first on any new dataset.

## 1. Columns That Contain a Single Value

> ***Zero-Variance Predictor:*** A column where every row has the exact same value; it carries no
> information for modeling and can also cause errors in some algorithms.

**Detection:** use the number of unique values per column (e.g., Pandas `nunique()`); a count of 1
flags the column for removal.

## 2. Columns With Very Few Unique Values

> ***Near-Zero-Variance Predictor:*** A column whose variance is not exactly zero but is very
> small — e.g., only 2 to 9 unique numeric values spread across thousands of rows.

These are **not automatically useless** — they may represent legitimate categorical/ordinal
information. **Detection:** compute the percentage of unique values per column
= (number of unique values ÷ total rows) × 100, and flag columns below a threshold
(commonly < 1%).

## 3. Columns With Low Variance

The statistical variance (average squared deviation from the mean) of a column can be used
directly as a filter — e.g., scikit-learn's `VarianceThreshold` removes columns whose variance
falls below a chosen cut-off.

## 4. Duplicate Rows

Identical rows add no new information and can bias a model or waste computation. Identify with
`duplicated()` and remove with `drop_duplicates()`.

**Exam tip:** Chapter 5 targets **uninformative columns/rows** (constant, near-constant,
duplicate) — this is distinct from handling **missing values** (Ch. 7-10) and **outliers** (Ch. 6).

In [ ]:
# 1. COLUMNS THAT CONTAIN A SINGLE VALUE ------------------------------------
counts = oil.nunique()
print("unique values per column (oil spill):")
print(counts.to_string())

single = [i for i, c in counts.items() if c == 1]
print("\nZero-variance columns to delete:", single)

oil_c = oil.drop(columns=single)
print("shape before:", oil.shape, "-> after:", oil_c.shape)

In [ ]:
# 2. COLUMNS WITH VERY FEW UNIQUE VALUES ------------------------------------
n_rows = oil.shape[0]
pct = (oil.nunique() / n_rows * 100)
for i, p in pct.items():
    if p < 1:
        print(f"col {i:>3}: {oil[i].nunique():>3} unique  ({p:.1f}%)")

few = [i for i, p in pct.items() if p < 1]
print("\nColumns below the 1% threshold:", few)
print("NOTE: review these by hand — some may be legitimate categorical/ordinal variables.")
oil_few = oil.drop(columns=few)
print("shape before:", oil.shape, "-> after:", oil_few.shape)

In [ ]:
# 3. COLUMNS WITH LOW VARIANCE (VarianceThreshold) --------------------------
from sklearn.feature_selection import VarianceThreshold

X_oil_df = oil.drop(columns=[OIL_TARGET])
for thresh in np.arange(0.0, 0.55, 0.05):
    n_kept = VarianceThreshold(threshold=thresh).fit_transform(X_oil_df).shape[1]
    print(f"threshold={thresh:.2f} -> features kept: {n_kept}")

vt = VarianceThreshold(threshold=0.0).fit(X_oil_df)
print("\nAt threshold 0.0 the removed (constant) columns are:",
      list(X_oil_df.columns[~vt.get_support()]))

In [ ]:
# 4. DUPLICATE ROWS ----------------------------------------------------------
demo = pd.concat([housing, housing.iloc[:5]], ignore_index=True)   # inject 5 duplicates
dups = demo.duplicated()
print("Any duplicates?", dups.any(), "| how many:", int(dups.sum()))
display(demo[dups].head())

clean = demo.drop_duplicates()
print("shape before:", demo.shape, "-> after drop_duplicates():", clean.shape)
print("\nReal datasets — duplicate rows: housing =", housing.duplicated().sum(),
      "| oil spill =", oil.duplicated().sum(), "| horse colic =", horse.duplicated().sum())

---
# Chapter 6 — Outlier Identification and Removal

> ***Outlier:*** An observation that lies an abnormal distance from other values in a dataset; an
> unlikely event under the assumed data-generating distribution. Causes include measurement
> variability, experimental error, natural rare events, and data-entry mistakes.

## (a) Standard Deviation Method — for Gaussian / near-Gaussian data

Empirical rule (68–95–99.7 rule) for a Gaussian distribution:

- 1 standard deviation from the mean → covers ≈ **68%** of the data.
- 2 standard deviations from the mean → covers ≈ **95%** of the data.
- 3 standard deviations from the mean → covers ≈ **99.7%** of the data.

> ***z-score:*** The number of standard deviations a value xᵢ is from the mean μ.
>
> **z = (xᵢ − μ) / σ**

Common rule of thumb: values with **|z| > 3** (outside mean ± 3σ) are flagged as outliers.
Use 2σ for smaller samples and 4σ for larger samples.

```
cut_off = std × 3
lower   = mean − cut_off
upper   = mean + cut_off
outlier if value < lower OR value > upper
```

## (b) Interquartile Range (IQR) Method — for non-Gaussian data

Percentiles/quartiles: sort the data; the 25th percentile (Q1), 50th percentile (median), and
75th percentile (Q3) divide it into four groups.

> ***IQR (Interquartile Range):*** IQR = Q3 − Q1 — represents the middle 50% (the "body") of the
> data; forms the box in a box-and-whisker plot.

```
lower = Q1 − k × IQR
upper = Q3 + k × IQR
```

- Common factor **k = 1.5** → ordinary outliers.
- Factor **k = 3** (or more) → "extreme outliers" / "far outs".

## (c) Automatic (Model-Based) Outlier Detection

Framed as **one-class classification**: the model characterizes what "normal" data looks like and
flags deviations.

> ***Local Outlier Factor (LOF):*** Scores each point by how isolated it is relative to the local
> density of its k-nearest neighbors; a larger score indicates a more likely outlier. Works well
> for low/moderate-dimensional feature spaces but degrades under the curse of dimensionality. In
> scikit-learn, `LocalOutlierFactor.fit_predict()` returns **+1 for inliers and −1 for outliers**.

Other model-based detectors: **Isolation Forest, One-Class SVM, Elliptic Envelope**.

## Multivariate Extension

For multiple Gaussian variables, the outlier boundary generalizes to an **ellipse/ellipsoid**
(standard-deviation method) or a **hyper-rectangle** (IQR method) in multi-dimensional space.

**Practical workflow:** fit the outlier detector on the **training set only** (to avoid data
leakage — same principle as Chapter 4), remove/mask flagged rows, then fit and evaluate the model.
This commonly improves error metrics such as **Mean Absolute Error (MAE)**.

In [ ]:
# (a) STANDARD DEVIATION METHOD ----------------------------------------------
data = housing["RM"].values          # average rooms per dwelling — near-Gaussian

mu, sd = data.mean(), data.std()
cut_off = sd * 3
lower, upper = mu - cut_off, mu + cut_off
print(f"mean={mu:.3f}  std={sd:.3f}  cut_off={cut_off:.3f}")
print(f"lower={lower:.3f}  upper={upper:.3f}")

outliers = data[(data < lower) | (data > upper)]
inliers  = data[(data >= lower) & (data <= upper)]
print(f"Identified outliers: {len(outliers)}  -> {np.round(outliers, 2)}")
print(f"Non-outlier observations: {len(inliers)}")

# equivalent z-score formulation
z = (data - mu) / sd
print(f"Same result via |z| > 3: {int((np.abs(z) > 3).sum())} outliers")

# empirical 68-95-99.7 check
for k in (1, 2, 3):
    print(f"within {k} sd: {np.mean(np.abs(z) <= k)*100:.1f}%")

In [ ]:
# (b) INTERQUARTILE RANGE (IQR) METHOD ---------------------------------------
data = housing["CRIM"].values        # heavily skewed — NOT Gaussian

q25, q50, q75 = np.percentile(data, [25, 50, 75])
iqr = q75 - q25
print(f"Q1={q25:.3f}  median={q50:.3f}  Q3={q75:.3f}  IQR={iqr:.3f}")

for k, label in [(1.5, "ordinary outliers"), (3.0, "extreme outliers / far outs")]:
    lo, hi = q25 - k * iqr, q75 + k * iqr
    flagged = data[(data < lo) | (data > hi)]
    print(f"k={k}: bounds=[{lo:.3f}, {hi:.3f}] -> {len(flagged)} {label}")

plt.boxplot([housing["CRIM"], housing["RM"], housing["LSTAT"]],
            tick_labels=["CRIM", "RM", "LSTAT"], whis=1.5)
plt.title("Box-and-whisker: box = IQR, whiskers = 1.5 x IQR, points beyond = outliers")
plt.tight_layout(); plt.show()

In [ ]:
# (c) AUTOMATIC / MODEL-BASED OUTLIER DETECTION ------------------------------
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import mean_absolute_error

Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=.33, random_state=1)
print("train shape before removal:", Xh_tr.shape, yh_tr.shape)

detectors = {
    "LocalOutlierFactor": LocalOutlierFactor(),                       # +1 inlier, -1 outlier
    "IsolationForest":    IsolationForest(contamination=.1, random_state=RANDOM_STATE),
    "OneClassSVM":        OneClassSVM(nu=.05),
    "EllipticEnvelope":   EllipticEnvelope(contamination=.05, random_state=RANDOM_STATE),
}

baseline = LinearRegression().fit(Xh_tr, yh_tr)
print(f"\nMAE with ALL training rows          : {mean_absolute_error(yh_te, baseline.predict(Xh_te)):.3f}")

for name, det in detectors.items():
    yhat = det.fit_predict(Xh_tr)          # fit on TRAINING data only -> no leakage
    mask = yhat != -1                      # keep inliers
    model = LinearRegression().fit(Xh_tr[mask], yh_tr[mask])
    mae = mean_absolute_error(yh_te, model.predict(Xh_te))
    print(f"MAE after {name:<20}: {mae:.3f}   (removed {int((~mask).sum()):3d} rows)")

In [ ]:
# Multivariate extension: the boundary becomes an ellipse (std-dev / Mahalanobis)
# or a hyper-rectangle (IQR), illustrated on two housing variables.
pair = housing[["RM", "LSTAT"]].values
ee = EllipticEnvelope(contamination=.05, random_state=RANDOM_STATE).fit(pair)
flag = ee.predict(pair)

xx, yy = np.meshgrid(np.linspace(3, 9, 200), np.linspace(0, 40, 200))
zz = ee.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contour(xx, yy, zz, levels=[0], colors="red")            # the ellipse boundary
plt.scatter(pair[flag == 1, 0], pair[flag == 1, 1], s=8, label="inlier")
plt.scatter(pair[flag == -1, 0], pair[flag == -1, 1], s=14, c="red", label="outlier")

q25, q75 = np.percentile(pair, [25, 75], axis=0); iqr2 = q75 - q25
lo, hi = q25 - 1.5*iqr2, q75 + 1.5*iqr2
plt.gca().add_patch(plt.Rectangle(lo, *(hi - lo), fill=False, ls="--", ec="green",
                                  label="IQR hyper-rectangle"))
plt.xlabel("RM"); plt.ylabel("LSTAT"); plt.legend(fontsize=8)
plt.title("Multivariate outlier boundaries: ellipse (Gaussian) vs rectangle (IQR)")
plt.tight_layout(); plt.show()

---
# Chapter 7 — How to Mark and Remove Missing Data

> ***Missing Data:*** Rows where one or more column values are absent, sometimes represented with
> a placeholder/sentinel character (e.g., "?", empty string, 0, "NULL") instead of being genuinely
> blank. Causes: unrecorded observations, data corruption, equipment malfunction, or merging
> incompatible datasets.

## Step 1 — Mark Missing Values

Replace sentinel/placeholder characters with a proper missing-value marker (**NaN**) — e.g.,
`DataFrame.replace('?', nan)` — so that pandas, NumPy, and scikit-learn correctly recognize them
as missing (checked with `isnull()` / `isna()`).

## Why Missing Values Are a Problem

Most machine learning algorithms **cannot operate on data containing NaN**, and will raise an
error when the model is fit — so missing values must always be handled (removed or imputed)
before modeling.

## Step 2 — Remove Rows With Missing Values (Listwise Deletion)

The simplest strategy: drop any row containing one or more NaN values, using `dropna()`.

**Downsides:** can discard a large fraction of the dataset (and potentially valuable information);
only viable if enough complete rows remain; not applicable when missingness is spread across most
rows.

*This limitation motivates the more sophisticated **Imputation** techniques covered in
Chapters 8–10 (filling in estimated values instead of discarding data).*

In [ ]:
# STEP 1 — MARK MISSING VALUES ----------------------------------------------
print("Raw horse-colic data — sentinels are the STRING '?', so pandas sees no missing values:")
display(horse_raw.head())
print("isnull() count on raw frame:", int(horse_raw.isnull().sum().sum()), " <- misleading!")

# Replace the sentinel with a proper NaN marker
marked = horse_raw.replace("?", np.nan).infer_objects(copy=False).astype(float)
# (equivalent one-liner at load time: pd.read_csv(url, header=None, na_values='?'))

print("\nAfter replace('?', nan) — missing values per column:")
print(marked.isnull().sum().to_string())
print("\nTotal missing:", int(marked.isnull().sum().sum()),
      f"({marked.isnull().mean().mean()*100:.1f}% of all cells)")

In [ ]:
# Which columns are worst affected?
pct_missing = (marked.isnull().mean() * 100).sort_values(ascending=False)
pct_missing[pct_missing > 0].plot(kind="bar", figsize=(9, 3))
plt.ylabel("% missing"); plt.xlabel("column index")
plt.title("Horse Colic — percentage of missing values per column")
plt.tight_layout(); plt.show()

In [ ]:
# WHY MISSING VALUES ARE A PROBLEM ------------------------------------------
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
Xm, ym = xy(marked, HORSE_TARGET)
try:
    cross_val_score(LinearDiscriminantAnalysis(), Xm, ym, cv=3)
except Exception as e:
    print("ERROR ->", type(e).__name__, ":", str(e).split("\n")[0][:130])

In [ ]:
# STEP 2 — REMOVE ROWS WITH MISSING VALUES (listwise deletion) --------------
print("shape before dropna():", marked.shape)
dropped_all = marked.dropna()
print("shape after  dropna():", dropped_all.shape)
print(f"=> only {len(dropped_all)} of {len(marked)} rows survive listwise deletion "
      f"({(1 - len(dropped_all)/len(marked))*100:.1f}% of the dataset LOST)")
print("\nThis dataset is an extreme illustration of the downside: with column 15 missing in 82%")
print("of rows, almost no row is complete, so dropna() is simply not viable here.")

# Practical compromise: drop the mostly-empty COLUMNS first, then drop the remaining rows.
keep_cols = [c for c in marked.columns if marked[c].isnull().mean() <= 0.50]
drop_cols = [c for c in marked.columns if c not in keep_cols]
pruned = marked[keep_cols].dropna()
print(f"\nDrop columns with >50% missing {drop_cols}, then dropna() -> {pruned.shape}")

Xd, yd = xy(pruned, HORSE_TARGET)
s_drop = cross_val_score(LinearDiscriminantAnalysis(), Xd, yd, scoring="accuracy",
                         cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3,
                                                    random_state=RANDOM_STATE))
print(f"\nLDA accuracy on the {len(pruned)} surviving rows: {s_drop.mean():.3f}")
print(f"It runs now — but on {len(pruned)/len(marked)*100:.0f}% of the data. Hence Chapters 8-10.")

---
# Chapter 8 — How to Use Statistical Imputation

> ***Data Imputation (Missing Data Imputation):*** Replacing missing values in a dataset with
> substituted, estimated values so that the complete dataset can be used with algorithms requiring
> no missing data.

> ***Statistical Imputation:*** Calculating a statistic per column (from the present, observed
> values in that column) and replacing all missing values in that column with the computed
> statistic.

## Common Statistics Used

- Column **mean**
- Column **median**
- Column **mode** (most frequent value)
- A **constant** value

## Scikit-Learn Implementation — `SimpleImputer`

The `strategy` parameter selects: `'mean'`, `'median'`, `'most_frequent'`, or `'constant'`.

**Leakage-safe workflow:** fit the imputer's statistics on the **TRAINING data only**, then
transform both train and test data (same principle as Chapter 4) — typically wrapped inside a
**Pipeline** together with the model for correct cross-validation.

Different strategies (mean/median/mode/constant) should be compared **empirically** for a given
dataset, since the best statistic to use is data-dependent.

In [ ]:
# SimpleImputer — basic usage ------------------------------------------------
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
Xt = imputer.fit_transform(Xm)

print("Missing BEFORE :", int(np.isnan(Xm).sum()))
print("Missing AFTER  :", int(np.isnan(Xt).sum()))
print("\nLearned per-column statistics (first 8):", np.round(imputer.statistics_[:8], 3))

# Look at one column before/after
c = 15   # the column with ~82% missing
print(f"\ncolumn {c}: observed mean = {np.nanmean(Xm[:, c]):.3f}")
print("rows 0-9 before:", np.round(Xm[:10, c], 2))
print("rows 0-9 after :", np.round(Xt[:10, c], 2))

In [ ]:
# Leakage-safe usage: imputer INSIDE a Pipeline, evaluated with cross-validation
from sklearn.ensemble import RandomForestClassifier

cv_s    = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=RANDOM_STATE)
cv_fast = RepeatedStratifiedKFold(n_splits=5,  n_repeats=1, random_state=RANDOM_STATE)  # for wide sweeps
pipe = Pipeline([("impute", SimpleImputer(strategy="mean")),
                 ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
s = cross_val_score(pipe, Xm, ym, scoring="accuracy", cv=cv_s, n_jobs=-1)
print(f"Mean-imputation pipeline accuracy: {s.mean():.3f} (+/- {s.std():.3f})")

In [ ]:
# Compare all four strategies empirically -----------------------------------
results, names = [], []
for strategy in ["mean", "median", "most_frequent", "constant"]:
    p = Pipeline([("impute", SimpleImputer(strategy=strategy)),
                  ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
    sc = cross_val_score(p, Xm, ym, scoring="accuracy", cv=cv_s, n_jobs=-1)
    results.append(sc); names.append(strategy)
    print(f"{strategy:>14}: {sc.mean():.3f} (+/- {sc.std():.3f})")

plt.boxplot(results, tick_labels=names, showmeans=True)
plt.ylabel("accuracy"); plt.title("Chapter 8 — SimpleImputer strategies compared")
plt.tight_layout(); plt.show()

---
# Chapter 9 — How to Use KNN Imputation

> ***KNN (Nearest Neighbor) Imputation:*** A model-based imputation approach where, for each
> missing value in a feature, a k-Nearest Neighbors model finds the k most similar training rows
> (based on a distance measure computed over the other features) and averages their values for
> that feature to estimate the missing value.

More sophisticated than statistical imputation because it exploits **relationships between
features** rather than relying on a single global column statistic.

## Configuration / Hyperparameters

- **Distance measure** (commonly Euclidean distance).
- **Number of neighbors, k.**

## Scikit-Learn Implementation — `KNNImputer`

Used similarly to `SimpleImputer` (fit / transform); must be fit **only on the training set** to
avoid data leakage, and is best wrapped inside a **Pipeline**. The choice of k should be
tuned/compared — a larger k is **not** always better for downstream model performance.

In [ ]:
# KNNImputer — basic usage ---------------------------------------------------
from sklearn.impute import KNNImputer

knn_imp = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")
Xk = knn_imp.fit_transform(Xm)
print("Missing BEFORE:", int(np.isnan(Xm).sum()), "-> AFTER:", int(np.isnan(Xk).sum()))

# Contrast with the single global statistic used in Chapter 8
c = 3
mask = np.isnan(Xm[:, c])
print(f"\ncolumn {c} — first 6 imputed values")
print("  mean imputation :", np.round(Xt[mask, c][:6], 3), "  (all identical)")
print("  KNN imputation  :", np.round(Xk[mask, c][:6], 3), "  (row-specific)")

In [ ]:
# Leakage-safe pipeline + comparison against statistical imputation ---------
pipe_knn = Pipeline([("impute", KNNImputer(n_neighbors=5)),
                     ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
s_knn = cross_val_score(pipe_knn, Xm, ym, scoring="accuracy", cv=cv_s, n_jobs=-1)
print(f"KNN imputation (k=5) accuracy : {s_knn.mean():.3f} (+/- {s_knn.std():.3f})")
print(f"Mean imputation accuracy      : {results[0].mean():.3f}")

In [ ]:
# Tune k — a larger k is NOT automatically better ---------------------------
k_results, k_names = [], []
for k in [1, 3, 5, 7, 15, 18, 21]:
    p = Pipeline([("impute", KNNImputer(n_neighbors=k)),
                  ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
    sc = cross_val_score(p, Xm, ym, scoring="accuracy", cv=cv_fast, n_jobs=-1)
    k_results.append(sc); k_names.append(str(k))
    print(f"k={k:>2}: {sc.mean():.3f} (+/- {sc.std():.3f})")

plt.boxplot(k_results, tick_labels=k_names, showmeans=True)
plt.xlabel("number of neighbours (k)"); plt.ylabel("accuracy")
plt.title("Chapter 9 — KNNImputer: effect of k")
plt.tight_layout(); plt.show()

---
# Chapter 10 — How to Use Iterative Imputation

> ***Iterative Imputation:*** Each feature with missing values is modeled as a function of all
> other features (a regression problem), and missing values are predicted feature-by-feature
> (sequentially), reusing previously imputed values as inputs for imputing subsequent features.
> The entire process is repeated (iterated) multiple times so that estimates for all features are
> progressively refined.

## Also Known As

- **Fully Conditional Specification (FCS)**
- **Multivariate Imputation by Chained Equations (MICE)**

## Key Configuration Details

- A **regression algorithm** (often a simple linear model) is fit per feature to predict its
  missing values from the other features.
- **Number of iterations:** typically kept small (commonly ≈10, sometimes 10–20) — a tunable
  hyperparameter.
- **Imputation order** (the sequence in which features are processed) is configurable — e.g.,
  ascending order of missing-value count — and can affect the result.

## Scikit-Learn Implementation — `IterativeImputer`

Located in `sklearn.impute` (requires enabling via
`sklearn.experimental.enable_iterative_imputer`). Used with fit/transform, fit on **training data
only**, and typically embedded in a **Pipeline** for correct cross-validation.

In [ ]:
# IterativeImputer — basic usage (note the required experimental import) ----
from sklearn.experimental import enable_iterative_imputer   # noqa: F401  <-- MUST come first
from sklearn.impute import IterativeImputer

it_imp = IterativeImputer(max_iter=10, random_state=RANDOM_STATE)   # default estimator: BayesianRidge
Xi = it_imp.fit_transform(Xm)
print("Missing BEFORE:", int(np.isnan(Xm).sum()), "-> AFTER:", int(np.isnan(Xi).sum()))

c = 3; mask = np.isnan(Xm[:, c])
print(f"\ncolumn {c} — first 6 imputed values")
print("  mean      :", np.round(Xt[mask, c][:6], 3))
print("  KNN       :", np.round(Xk[mask, c][:6], 3))
print("  iterative :", np.round(Xi[mask, c][:6], 3))

In [ ]:
# Leakage-safe pipeline evaluation -------------------------------------------
pipe_it = Pipeline([("impute", IterativeImputer(random_state=RANDOM_STATE)),
                    ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
s_it = cross_val_score(pipe_it, Xm, ym, scoring="accuracy", cv=cv_s, n_jobs=-1)
print(f"Iterative imputation accuracy: {s_it.mean():.3f} (+/- {s_it.std():.3f})")

In [ ]:
# Configuration 1: the per-feature regression algorithm ---------------------
from sklearn.linear_model import BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

for est in [BayesianRidge(), DecisionTreeRegressor(random_state=RANDOM_STATE),
            ExtraTreesRegressor(n_estimators=10, random_state=RANDOM_STATE),
            KNeighborsRegressor(n_neighbors=5)]:
    p = Pipeline([("impute", IterativeImputer(estimator=est, random_state=RANDOM_STATE)),
                  ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
    sc = cross_val_score(p, Xm, ym, scoring="accuracy", cv=cv_fast, n_jobs=-1)
    print(f"{type(est).__name__:<22}: {sc.mean():.3f} (+/- {sc.std():.3f})")

In [ ]:
# Configuration 2: number of iterations --------------------------------------
it_results, it_names = [], []
for m in [1, 2, 3, 4, 5, 10, 15, 20]:
    p = Pipeline([("impute", IterativeImputer(max_iter=m, random_state=RANDOM_STATE)),
                  ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
    sc = cross_val_score(p, Xm, ym, scoring="accuracy", cv=cv_fast, n_jobs=-1)
    it_results.append(sc); it_names.append(str(m))
    print(f"max_iter={m:>2}: {sc.mean():.3f}")

plt.boxplot(it_results, tick_labels=it_names, showmeans=True)
plt.xlabel("max_iter"); plt.ylabel("accuracy")
plt.title("Chapter 10 — effect of the number of iterations")
plt.tight_layout(); plt.show()

In [ ]:
# Configuration 3: imputation order ------------------------------------------
for order in ["ascending", "descending", "roman", "arabic", "random"]:
    p = Pipeline([("impute", IterativeImputer(imputation_order=order, random_state=RANDOM_STATE)),
                  ("model",  RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])
    sc = cross_val_score(p, Xm, ym, scoring="accuracy", cv=cv_fast, n_jobs=-1)
    print(f"imputation_order={order:<11}: {sc.mean():.3f} (+/- {sc.std():.3f})")
print("\n('ascending' = fewest missing values first; order can change the result.)")

## Comparing the Three Imputation Approaches (Chapters 8–10)

| Method | Basis | Chapter |
|---|---|---|
| **Statistical Imputation** | Single column statistic: mean / median / mode / constant | 8 |
| **KNN Imputation** | Average of the k nearest-neighbor rows | 9 |
| **Iterative Imputation (MICE)** | Regression model per feature, iterated across all features | 10 |

In [ ]:
# Head-to-head on the same data, same model, same CV -------------------------
comparison = {
    "Ch7 dropna (52 rows left)": None,
    "Ch8 SimpleImputer(mean)": SimpleImputer(strategy="mean"),
    "Ch8 SimpleImputer(median)": SimpleImputer(strategy="median"),
    "Ch9 KNNImputer(k=5)": KNNImputer(n_neighbors=5),
    "Ch10 IterativeImputer": IterativeImputer(random_state=RANDOM_STATE),
}
scores_all, labels = [], []
for label, imp in comparison.items():
    if imp is None:   # listwise deletion: a much smaller dataset, so scores are only indicative
        sc = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
                             Xd, yd, scoring="accuracy", cv=cv_fast, n_jobs=-1)
    else:
        sc = cross_val_score(Pipeline([("impute", imp),
                                       ("model", RandomForestClassifier(n_estimators=100,
                                                                        random_state=RANDOM_STATE))]),
                             Xm, ym, scoring="accuracy", cv=cv_s, n_jobs=-1)
    scores_all.append(sc); labels.append(label)
    print(f"{label:<28}: {sc.mean():.3f} (+/- {sc.std():.3f})")

plt.figure(figsize=(9, 4))
plt.boxplot(scores_all, tick_labels=[l.replace(" ", "\n", 1) for l in labels], showmeans=True)
plt.ylabel("accuracy"); plt.title("Chapters 7-10 — handling missing data, head to head")
plt.xticks(fontsize=8); plt.tight_layout(); plt.show()

---
# Quick-Revision Summary — All 10 Chapters

| Ch | Topic | Must-Know Definition / Algorithm |
|---|---|---|
| **1** | ML Process & Data Prep | 4-step process: Define → Prepare → Evaluate → Finalize. Data preparation = transforming raw data into a form suitable for modeling. |
| **2** | Importance of Data Prep | Data must be numeric and meet algorithm requirements; model performance depends heavily on data representation. |
| **3** | Taxonomy | 5 tasks: Cleaning; Feature Selection (filter / wrapper / intrinsic); Transforms (normalize, standardize, power, quantile, one-hot, ordinal, discretize); Feature Engineering (polynomial); Dimensionality Reduction (PCA, SVD, LDA). |
| **4** | Data Leakage | Fit every data-prep step on training data only; use a Pipeline inside cross-validation. |
| **5** | Basic Cleaning | Remove zero-variance columns; examine near-zero-variance columns; remove duplicate rows. |
| **6** | Outliers | Std-Dev method (z-score, 3σ rule); IQR method (Q1 − 1.5×IQR, Q3 + 1.5×IQR); LOF / Isolation Forest for automatic detection. |
| **7** | Missing Data | Mark sentinel characters as NaN; listwise deletion (dropna) is the simplest fix. |
| **8** | Statistical Imputation | `SimpleImputer` — mean / median / mode / constant. |
| **9** | KNN Imputation | `KNNImputer` — average of k nearest neighbors. |
| **10** | Iterative Imputation | `IterativeImputer` / MICE — per-feature regression, repeated iterations. |

> **Golden Rule (Chapters 4–10):** Every data preparation step (scaling, imputing, outlier
> removal, feature selection) must be fit on **training data only**, then applied to test data —
> never fit on the full dataset before splitting, or you introduce **data leakage**.

In [ ]:
# The Golden Rule as one reusable, end-to-end, leakage-free template ---------
from sklearn.pipeline import Pipeline

golden = Pipeline([
    ("impute",  IterativeImputer(random_state=RANDOM_STATE)),   # Ch 8-10
    ("select",  SelectKBest(score_func=f_classif, k=15)),       # Ch 3.2
    ("scale",   StandardScaler()),                              # Ch 3.3
    ("reduce",  PCA(n_components=10)),                          # Ch 3.5
    ("model",   RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
])

s = cross_val_score(golden, Xm, ym, scoring="accuracy", cv=cv_s, n_jobs=-1)
print(f"Full leakage-free pipeline accuracy: {s.mean():.3f} (+/- {s.std():.3f})")
print("\nEvery step above is re-fit on each fold's TRAINING portion only.")
print("This is pipeline evaluation, not merely model evaluation.")

---
# Appendix A — Export This Notebook to a Styled PDF - Convert markdown to Python cells

Everything below turns this notebook into a **print-quality PDF**, with the page **border length
(margins), border frame, and font styles fully configurable** from the single `PDF_CONFIG`
dictionary in the next cell. Change a value, re-run the last cell, get a new PDF.

**How it works**

1. `nbconvert` renders the notebook (markdown + code + outputs, images inlined as base64) to a
   minimal HTML body.
2. A stylesheet is **generated from `PDF_CONFIG`** — `@page` controls the paper size, the four
   margins ("border length") and an optional printed frame; the font block controls body, heading
   and code typefaces.
3. An HTML→PDF engine renders it. Engines are tried in order and the first one available wins:
   **WeasyPrint** (best CSS support: real `@page` margins, page-border frame, page numbers) →
   **wkhtmltopdf** (CLI margins) → **xhtml2pdf** (pure Python).
   The styled `.html` is always written too, so you can open it and use *Print → Save as PDF*
   in any browser as a zero-dependency fallback.

**One-time install** (only what you don't already have):

```bash
pip install nbconvert weasyprint          # preferred
# or:  pip install xhtml2pdf              # pure python, no system libraries
# or:  apt-get install wkhtmltopdf        # CLI renderer
```

> **Save the notebook first** (`Ctrl/Cmd + S`) — the exporter reads the `.ipynb` file from disk,
> not the live in-memory kernel state.

## A.1 — The configuration dictionary

This is the only cell you need to edit. Every key is documented inline.

NOTEBOOK_FILE = "Data_Preparation_for_ML_Ch1-10.ipynb"   # <- name of THIS notebook on disk
OUTPUT_PDF    = "Data_Preparation_for_ML_Ch1-10.pdf"

PDF_CONFIG = {
    # ---- PAGE GEOMETRY / "BORDER LENGTH" -----------------------------------
    "page_size":        "A4",        # 'A4' | 'Letter' | 'Legal' | 'A3'
    "orientation":      "portrait",  # 'portrait' | 'landscape'
    "margin_top_mm":    18,          # <-- the four "border lengths" of the page
    "margin_bottom_mm": 18,
    "margin_left_mm":   20,
    "margin_right_mm":  16,

    # ---- PRINTED BORDER FRAME (the visible rectangle around the text) ------
    "border_show":      True,
    "border_width_mm":  0.5,         # thickness of the frame
    "border_style":     "solid",     # 'solid' | 'double' | 'dashed' | 'dotted' | 'none'
    "border_color":     "#25455f",
    "border_padding_mm": 6,          # gap between the frame and the text

    # ---- FONTS -------------------------------------------------------------
    "body_font":        "Georgia, 'Times New Roman', 'DejaVu Serif', serif",
    "heading_font":     "'Helvetica Neue', Helvetica, Arial, 'DejaVu Sans', sans-serif",
    "code_font":        "'DejaVu Sans Mono', 'Courier New', monospace",
    "body_size_pt":     10.0,
    "code_size_pt":     8.0,
    "h1_size_pt":       19.0,
    "h2_size_pt":       14.0,
    "h3_size_pt":       11.5,
    "line_height":      1.45,
    "text_color":       "#1b1b1b",
    "heading_color":    "#12395c",
    "code_bg":          "#f6f8fa",
    "code_style":       "friendly",   # pygments syntax-highlight theme: friendly|default|tango|bw
    "quote_bg":         "#eef4fa",

    # ---- CONTENT / LAYOUT --------------------------------------------------
    "include_code":     True,        # False -> a clean "notes only" revision PDF
    "include_outputs":  True,        # False -> hide printed output and figures
    "page_numbers":     True,
    "footer_text":      "Data Preparation for Machine Learning — Ch. 1-10",
    "chapter_page_breaks": True,     # start every H1 (chapter) on a new page
    "max_figure_width_pct": 92,      # scale plots to this % of the text width

    # ---- ENGINE ------------------------------------------------------------
    "engine": "auto",                # 'auto' | 'weasyprint' | 'wkhtmltopdf' | 'xhtml2pdf'
}

# Handy presets — assign one of these into PDF_CONFIG to switch look instantly.
PRESETS = {
    "classic_textbook": dict(body_font="Georgia, 'Times New Roman', serif",
                             heading_font="Georgia, 'Times New Roman', serif",
                             body_size_pt=10.5, margin_left_mm=25, margin_right_mm=25,
                             border_show=False),
    "wide_margin_notes": dict(margin_left_mm=32, margin_right_mm=12, border_show=True,
                              border_style="dashed", body_size_pt=10.0),
    "compact_revision":  dict(body_size_pt=8.5, code_size_pt=7.0, line_height=1.25,
                              margin_top_mm=10, margin_bottom_mm=10,
                              margin_left_mm=10, margin_right_mm=10, include_code=False),
    "modern_sans":       dict(body_font="'Helvetica Neue', Arial, sans-serif",
                              heading_font="'Helvetica Neue', Arial, sans-serif",
                              heading_color="#0f766e", border_color="#0f766e",
                              border_style="double"),
}
# Example:  PDF_CONFIG.update(PRESETS["modern_sans"])

print("Configured page:", PDF_CONFIG["page_size"], PDF_CONFIG["orientation"])
print("Margins (mm) T/B/L/R:", PDF_CONFIG["margin_top_mm"], PDF_CONFIG["margin_bottom_mm"],
      PDF_CONFIG["margin_left_mm"], PDF_CONFIG["margin_right_mm"])

## A.2 — Which fonts can I use?

A PDF engine can only use fonts installed on the machine. This lists what is available so you can
paste a real family name into `body_font` / `heading_font` / `code_font`.

from matplotlib import font_manager

fams = sorted({f.name for f in font_manager.fontManager.ttflist})
print(f"{len(fams)} font families available. A selection:")
print(", ".join(fams[:40]))
mono = [f for f in fams if any(k in f.lower() for k in ("mono", "courier", "consol"))]
print("\nMonospace families (for code_font):", ", ".join(mono) if mono else "none found")
print("\nTip: always end a font stack with a generic fallback, e.g. \"'My Font', serif\".")

## A.3 — The exporter

Two functions: `build_css(config)` turns the dictionary into a stylesheet, and
`notebook_to_pdf(...)` runs the conversion. Nothing else needs editing.

import os, shutil, subprocess

PAGE_SIZES = {"A4": "A4", "Letter": "Letter", "Legal": "Legal", "A3": "A3"}


def build_css(cfg):
    """Translate the PDF_CONFIG dictionary into a print stylesheet."""
    size = f'{PAGE_SIZES.get(cfg["page_size"], "A4")} {cfg["orientation"]}'
    margins = (f'{cfg["margin_top_mm"]}mm {cfg["margin_right_mm"]}mm '
               f'{cfg["margin_bottom_mm"]}mm {cfg["margin_left_mm"]}mm')

    if cfg["border_show"] and cfg["border_style"] != "none":
        frame = (f'border: {cfg["border_width_mm"]}mm {cfg["border_style"]} {cfg["border_color"]};'
                 f' padding: {cfg["border_padding_mm"]}mm;')
    else:
        frame = ""

    footer = ""
    if cfg["page_numbers"]:
        footer = ('@bottom-center { content: "' + cfg["footer_text"] +
                  '   |   page " counter(page) " of " counter(pages);'
                  ' font-size: 7.5pt; color: #666; }')

    page_break = ("h1 { page-break-before: always; }"
                  if cfg["chapter_page_breaks"] else "")
    hide_in = (".jp-InputArea, div.input, .jp-Cell-inputWrapper { display: none !important; }"
               if not cfg["include_code"] else "")
    hide_out = (".jp-OutputArea, div.output_wrapper, .jp-Cell-outputWrapper { display: none !important; }"
                if not cfg["include_outputs"] else "")

    return f"""
@page {{ size: {size}; margin: {margins}; {frame} {footer} }}

html, body {{ font-family: {cfg["body_font"]}; font-size: {cfg["body_size_pt"]}pt;
              line-height: {cfg["line_height"]}; color: {cfg["text_color"]};
              background: #fff; margin: 0; padding: 0; }}

h1, h2, h3, h4, h5 {{ font-family: {cfg["heading_font"]}; color: {cfg["heading_color"]};
                      line-height: 1.2; page-break-after: avoid; margin: 0.9em 0 0.35em; }}
h1 {{ font-size: {cfg["h1_size_pt"]}pt; border-bottom: 1.6pt solid {cfg["heading_color"]};
      padding-bottom: 3pt; }}
h2 {{ font-size: {cfg["h2_size_pt"]}pt; }}
h3 {{ font-size: {cfg["h3_size_pt"]}pt; }}
h4 {{ font-size: {cfg["body_size_pt"] + 0.5}pt; }}
{page_break}

p, li {{ orphans: 3; widows: 3; }}
blockquote {{ background: {cfg["quote_bg"]}; border-left: 3pt solid {cfg["heading_color"]};
              margin: 0.6em 0; padding: 0.45em 0.8em; page-break-inside: avoid; }}

pre, code, .highlight, .input_area {{ font-family: {cfg["code_font"]};
                                      font-size: {cfg["code_size_pt"]}pt; }}
div.input_area, .jp-InputArea-editor, .highlight pre {{
    background: {cfg["code_bg"]}; border: 0.4pt solid #d5dbe1; border-radius: 3pt;
    padding: 5pt 7pt; page-break-inside: avoid;
    white-space: pre-wrap; word-wrap: break-word; }}
.output_subarea pre, .jp-OutputArea-output pre {{
    background: #fbfbfb; border-left: 2pt solid #b9c4cf; padding: 4pt 7pt;
    white-space: pre-wrap; word-wrap: break-word; }}
.output_stderr {{ display: none; }}

img {{ max-width: {cfg["max_figure_width_pct"]}%; height: auto;
       display: block; margin: 6pt auto; page-break-inside: avoid; }}

table {{ border-collapse: collapse; margin: 0.6em 0; font-size: {cfg["body_size_pt"] - 1}pt;
         page-break-inside: avoid; width: 100%; }}
th, td {{ border: 0.4pt solid #b9c4cf; padding: 3pt 5pt; text-align: left; vertical-align: top; }}
thead th {{ background: {cfg["quote_bg"]}; font-family: {cfg["heading_font"]};
            color: {cfg["heading_color"]}; }}
hr {{ border: none; border-top: 0.6pt solid #c9d2da; margin: 1em 0; }}
.prompt, .jp-InputPrompt, .jp-OutputPrompt {{ display: none; }}
.anchor-link {{ display: none; }}
.jp-Cell, .cell {{ page-break-inside: auto; }}
{hide_in}
{hide_out}
"""


def notebook_to_pdf(notebook=None, output=None, config=None, keep_html=True, verbose=True):
    """Convert a .ipynb into a styled PDF using the settings in `config`.

    config keys override PDF_CONFIG for this call only, e.g.
        notebook_to_pdf(config={"margin_left_mm": 35, "body_font": "Arial, sans-serif"})
    """
    cfg = dict(PDF_CONFIG)
    cfg.update(config or {})
    notebook = notebook or NOTEBOOK_FILE
    output = output or OUTPUT_PDF

    if not os.path.exists(notebook):
        raise FileNotFoundError(f"{notebook} not found - save the notebook first, "
                                f"or set NOTEBOOK_FILE to its real name.")

    # 1) notebook -> minimal HTML body ---------------------------------------
    import nbformat
    from nbconvert import HTMLExporter

    nb = nbformat.read(notebook, as_version=4)
    exporter = HTMLExporter(template_name="basic")
    exporter.exclude_input = not cfg["include_code"]
    exporter.exclude_output = not cfg["include_outputs"]
    body, _ = exporter.from_notebook_node(nb)

    # the very first heading must not start on a fresh page
    body = body.replace("<h1", '<h1 style="page-break-before:avoid"', 1)

    # syntax highlighting for the code cells
    try:
        from pygments.formatters import HtmlFormatter
        pygments_css = HtmlFormatter(style=cfg.get("code_style", "friendly")).get_style_defs(".highlight")
    except Exception:
        pygments_css = ""

    html = ('<!DOCTYPE html><html><head><meta charset="utf-8">'
            f"<style>{pygments_css}</style>"
            f"<style>{build_css(cfg)}</style></head><body>{body}</body></html>")

    html_path = os.path.splitext(output)[0] + ".html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    if verbose:
        print(f"[1/2] styled HTML written -> {html_path}")

    # 2) HTML -> PDF, trying engines in order --------------------------------
    order = (["weasyprint", "wkhtmltopdf", "xhtml2pdf"]
             if cfg["engine"] == "auto" else [cfg["engine"]])
    errors = []
    for engine in order:
        try:
            if engine == "weasyprint":
                from weasyprint import HTML
                HTML(string=html, base_url=".").write_pdf(output)

            elif engine == "wkhtmltopdf":
                if shutil.which("wkhtmltopdf") is None:
                    raise RuntimeError("wkhtmltopdf not on PATH")
                subprocess.run(["wkhtmltopdf", "--quiet", "--enable-local-file-access",
                                "--page-size", cfg["page_size"],
                                "--orientation", cfg["orientation"].capitalize(),
                                "-T", f'{cfg["margin_top_mm"]}mm',
                                "-B", f'{cfg["margin_bottom_mm"]}mm',
                                "-L", f'{cfg["margin_left_mm"]}mm',
                                "-R", f'{cfg["margin_right_mm"]}mm',
                                html_path, output], check=True)

            elif engine == "xhtml2pdf":
                from xhtml2pdf import pisa
                with open(output, "wb") as fh:
                    status = pisa.CreatePDF(html, dest=fh)
                if status.err:
                    raise RuntimeError("xhtml2pdf reported errors")
            else:
                raise ValueError(f"unknown engine '{engine}'")

            size_kb = os.path.getsize(output) / 1024
            if verbose:
                print(f"[2/2] PDF written with {engine} -> {output}  ({size_kb:.0f} KB)")
            if not keep_html:
                os.remove(html_path)
            return output

        except Exception as e:
            errors.append(f"  - {engine}: {type(e).__name__}: {str(e)[:120]}")

    print("No PDF engine succeeded:")
    print("\n".join(errors))
    print(f"\nFallback: open {html_path} in a browser and use Print -> Save as PDF "
          f"(page size, margins and fonts are already baked into the CSS).")
    return html_path


print("Exporter ready: build_css() and notebook_to_pdf() are defined.")

## A.4 — Generate the PDF

Run this cell. To restyle, edit `PDF_CONFIG` (or apply a preset) and run it again — or pass
overrides directly as keyword-style entries in the `config` argument, as the examples show.

# Default styling
notebook_to_pdf()

# --- Variations: uncomment any of these -------------------------------------
# Wider left border for hand annotations, dashed frame:
# notebook_to_pdf(output="notes_wide_margin.pdf",
#                 config={"margin_left_mm": 35, "border_style": "dashed"})

# Notes-only revision booklet (no code, no outputs), tight borders, sans-serif:
# notebook_to_pdf(output="revision_notes_only.pdf",
#                 config={"include_code": False, "include_outputs": False,
#                         "body_font": "'Helvetica Neue', Arial, sans-serif",
#                         "body_size_pt": 9.0, "margin_top_mm": 12, "margin_bottom_mm": 12,
#                         "margin_left_mm": 12, "margin_right_mm": 12})

# US Letter, no printed frame, larger serif type:
# notebook_to_pdf(output="letter_large_print.pdf",
#                 config={"page_size": "Letter", "border_show": False, "body_size_pt": 12.0})

# Apply a named preset:
# cfg = dict(PDF_CONFIG); cfg.update(PRESETS["compact_revision"])
# notebook_to_pdf(output="compact.pdf", config=cfg)

---

### Appendix B — Alternative export routes (if you prefer)

| Route | Command | Notes |
|---|---|---|
| LaTeX (highest typographic quality) | `jupyter nbconvert --to pdf notebook.ipynb` | Needs a TeX distribution; control margins with a custom template setting `\usepackage[margin=20mm]{geometry}`. |
| Headless Chrome | `jupyter nbconvert --to webpdf --allow-chromium-download notebook.ipynb` | Excellent CSS support; margins via the same `@page` rule used above. |
| Browser print | open the generated `.html` → *Print → Save as PDF* | Zero extra dependencies; the CSS above already sets size, margins and fonts. |

---

**End of notes — Data Preparation for Machine Learning, Chapters 1–10.**